# Trabalho final: Modelos de recomendaÃ§Ã£o e anÃ¡lise de espaÃ§os latentes

**Disciplina:** *Aprendizado nÃ£o-supervisionado* - *MC847*\
**InstituiÃ§Ã£o:** *Universidade Estadual de Campinas*\
**Professor(a):** *Marcos Medeiros Raimundo*\
**Semestre:** *2s2026*

| Nome | RA |
|---|---|
| *Integrante 1* | *RA 1* |
| *Integrante 2* | *RA 2* |
| *Integrante 3 (opcional)* | *RA 3* |

Preencha a tabela acima com os nomes e RAs dos integrantes da equipe (2 ou 3 alunos, impreterivelmente).

## IntroduÃ§Ã£o

Em sistemas de recomendaÃ§Ã£o, a principal tarefa Ã© aprender a preferÃªncia de um usuÃ¡rio a partir de um histÃ³rico parcial de interaÃ§Ãµes.
Isso pode ser visto como um problema de *completar informaÃ§Ã£o*: dado o histÃ³rico parcial de um usuÃ¡rio, quais itens provavelmente faltam?
Neste trabalho vocÃªs vÃ£o prever notas nÃ£o observadas com modelos que fazem suposiÃ§Ãµes diferentes sobre essa estrutura e, depois, investigar o que o espaÃ§o aprendido contÃ©m e quais suposiÃ§Ãµes o produziram.

O trabalho Ã© dividido em trÃªs etapas, sendo as duas primeiras comuns a todos os grupos e a terceira a ser definida por cada grupo.
A primeira etapa consiste em definir uma baseline que serÃ¡ usada como referÃªncia para as demais etapas.
A segunda consiste em realizar e analisar experimentos para entender como diferentes suposiÃ§Ãµes sobre a estrutura latente dos dados afetam o desempenho do modelo.
A Ãºltima etapa Ã© a contribuiÃ§Ã£o de cada grupo para o problema, que pode ser uma proposta de melhoria do modelo ou uma anÃ¡lise mais profunda de algum aspecto do problema.

As duas primeiras etapas tÃªm cÃ³digos de referÃªncia, que podem ser usados como base para a implementaÃ§Ã£o dos modelos.
VocÃªs podem usar bibliotecas externas com implementaÃ§Ãµes de modelos, mas devem deixar claro quais bibliotecas foram usadas e como elas foram usadas.
Por exemplo, vocÃªs podem reescrever o modelo de fatoraÃ§Ã£o de matrizes em PyTorch ou Jax para melhorar a performance, desde que justifiquem a escolha e expliquem como a implementaÃ§Ã£o foi mantida fiel ao modelo original.

## Dataset

Utilizaremos o conjunto de dados MovieLens 100K [(Harper e Konstan, 2015)](https://www.scirp.org/reference/referencespapers?referenceid=3586299): 100 mil notas inteiras de 1 a 5, dadas por 943 usuÃ¡rios a 1.682 filmes, com pelo menos 20 notas por usuÃ¡rio.
A matriz tem densidade de cerca de 6%, cada usuÃ¡rio tem idade (em anos), gÃªnero e ocupaÃ§Ã£o (21 categorias) e cada filme tem tÃ­tulo, ano e 19 indicadores de gÃªnero, e um filme pode ter vÃ¡rios gÃªneros.

Devido Ã  natureza do problema, a ausÃªncia de notas nÃ£o pode ser tratada como aleatÃ³ria: Os motivos pelos quais um usuÃ¡rio nÃ£o avaliou um item podem ser variados, incluindo falta de interesse, falta de exposiÃ§Ã£o ao item ou simplesmente esquecimento.
Isso significa que os dados de treinamento sÃ£o enviesados e nÃ£o representam a distribuiÃ§Ã£o real de preferÃªncias dos usuÃ¡rios.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from numpy.typing import NDArray

sns.set_theme(style="whitegrid")

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
DATA_DIR = "./data/"

def set_seed(seed: int) -> None:
    """Set the random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
from pandas.core.frame import DataFrame

GENRES = [
    "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]


def load_movielens() -> tuple[DataFrame, DataFrame, DataFrame]:
    """Download and extract the MovieLens dataset."""
    import zipfile

    import requests

    data_dir = Path(DATA_DIR)
    data_dir.mkdir(parents=True, exist_ok=True)

    zip_path = data_dir / "ml-100k.zip"
    target_dir = data_dir / "ml-100k"

    if not target_dir.exists():
        if not zip_path.exists():
            print("Downloading MovieLens dataset...")
            response = requests.get(MOVIELENS_URL)

            with open(zip_path, "wb") as f:
                f.write(response.content)

            print("Download complete.")

        print("Extracting MovieLens dataset...")
        with zipfile.ZipFile(zip_path) as zip_ref:
            zip_ref.extractall(data_dir)

        print(f"Extracted to {data_dir}.")

    ratings = pd.read_csv(
            target_dir / "u.data",
            sep="\t",
            names=["user_id", "item_id", "rating", "timestamp"],
            dtype={"user_id": np.int32, "item_id": np.int32, "rating": np.float32, "timestamp": np.int64},
        )

    users = pd.read_csv(
            target_dir / "u.user",
            sep="|",
            names=["user_id", "age", "gender", "occupation", "zip_code"],
            dtype={"user_id": np.int32, "age": np.int32, "gender": str, "occupation": str, "zip_code": str},
        )

    item_cols = ["item_id", "title", "release_date", "video_release_date", "imdb_url", *GENRES]

    movies = pd.read_csv(
        target_dir / "u.item",
        sep="|",
        encoding="latin-1",
        names=item_cols,
        usecols=["item_id", "title", "release_date", *GENRES],
        dtype={"item_id": np.int32, "title": str, "release_date": str},
    )

    movies["year"] = pd.to_numeric(movies["title"].str.extract(r"\((\d{4})\)")[0], errors="coerce")

    movies["genres"] = movies[GENRES].apply(
        lambda row: "|".join([g for g in GENRES if row[g] == 1]), axis=1
    )

    # NormalizaÃ§Ã£o para que os IDs de usuÃ¡rio e item comecem em 0
    ratings["user_id"] -= 1
    ratings["item_id"] -= 1
    users["user_id"] -= 1
    movies["item_id"] -= 1

    return ratings, users, movies


def ratings_to_matrix(ratings: pd.DataFrame, fill_value: float = float("nan")) -> NDArray[np.float64]:
    """Convert ratings DataFrame to a user-item matrix."""
    num_users = ratings["user_id"].max() + 1
    num_items = ratings["item_id"].max() + 1

    user_item_matrix = np.full((num_users, num_items), fill_value, dtype=np.float64)
    user_item_matrix[ratings["user_id"], ratings["item_id"]] = ratings["rating"].values.astype(np.float64)

    return user_item_matrix


In [ ]:
ratings, users, movies = load_movielens()
density = len(ratings) / (len(users) * len(movies))
print(f"notas={len(ratings):,}; usuÃ¡rios={len(users):,}; filmes={len(movies):,}; densidade={density:.3f}")

## FormulaÃ§Ã£o do problema

Uma suposiÃ§Ã£o comum em sistemas de recomendaÃ§Ã£o Ã© que a preferÃªncia de um usuÃ¡rio por um item pode ser representada como uma funÃ§Ã£o sobre um espaÃ§o latente compartilhado entre usuÃ¡rios e itens.
Ou seja, considerando que cada usuÃ¡rio e cada item podem ser representados como vetores em um espaÃ§o latente, a nota de um usuÃ¡rio para um item pode ser modelada como uma funÃ§Ã£o que mede a compatibilidade entre os vetores do usuÃ¡rio e do item.

Utilizaremos uma formulaÃ§Ã£o geral para este problema, que pode acomodar diferentes modelos e suposiÃ§Ãµes sobre como os vetores latentes sÃ£o gerados e como a funÃ§Ã£o de compatibilidade Ã© definida.
Considere um conjunto de usuÃ¡rios $U$ e um conjunto de itens $I$, e seja $R$ a matriz de notas, a nota do usuÃ¡rio $u$ para o item $i$, $R_{ui}$ Ã© modelada como:

$$
R_{ui} = \mu + b_u + b_i + f(\mathbf{z}_u, \mathbf{w}_i) + \epsilon_{ui}
$$
onde $\mu$ Ã© a mÃ©dia global das notas, $b_u$ e $b_i$ sÃ£o os *biases* do usuÃ¡rio e do item, respectivamente, $\mathbf{z}_u$ e $\mathbf{w}_i$ sÃ£o os vetores latentes do usuÃ¡rio e do item, $f$ Ã© a funÃ§Ã£o de compatibilidade entre os vetores latentes e $\epsilon_{ui}$ Ã© o erro de modelagem para aquele par usuÃ¡rio-item.

Um desafio importante Ã© que a matriz de notas $R$ Ã© muito esparsa, ou seja, grande parte dos pares usuÃ¡rio-item nÃ£o possuem notas observadas. Isso significa que aprender a estrutura latente compartilhada pode ser difÃ­cil sem fazer suposiÃ§Ãµes adicionais sobre como os vetores latentes sÃ£o gerados e como a funÃ§Ã£o de compatibilidade Ã© definida.

Vamos explorar como diferentes suposiÃ§Ãµes afetam o desempenho do modelo e o que podemos aprender sobre a estrutura latente dos dados atravÃ©s da inspeÃ§Ã£o do espaÃ§o latente aprendido.

> **Importante:** Os modelos tÃªm uma abordagem transdutiva, ou seja, eles aprendem os vetores latentes apenas para os usuÃ¡rios e itens presentes no conjunto de treinamento.
As features de um usuÃ¡rio nÃ£o sÃ£o as notas que ele deu, mas sim apenas um identificador Ãºnico, e o mesmo vale para os itens.
Isso permite que o modelo possua parÃ¢metros especÃ­ficos para cada usuÃ¡rio, mas tambÃ©m significa que ele nÃ£o pode generalizar para novos usuÃ¡rios ou itens sem re-treinamento.

## Diretrizes para o trabalho

Para cada experimento, sigam as seguintes diretrizes:

> 1. Escrevam a hipÃ³tese e a previsÃ£o **antes** de rodar cada experimento.
> 2. Resultados negativos ou inconclusivos valem, se vierem com **anÃ¡lise crÃ­tica**.
> 3. Todos os modelos, inclusive os mais simples, passam pelo mesmo protocolo (a seguir).
> 4. Separem o que um resultado diz sobre a previsÃ£o do que ele diz sobre a representaÃ§Ã£o latente.
> 5. Citem fontes sempre que necessÃ¡rio, seja para motivar uma escolha, ou para justificar uma decisÃ£o.

A nota nÃ£o depende da mÃ©trica obtida pelo modelo, mas da qualidade das hipÃ³teses, aplicaÃ§Ã£o correta do protocolo e da anÃ¡lise das evidÃªncias a favor, contra ou insuficientes.

### Protocolo experimental

Os experimentos devem ser controlados, com uma variÃ¡vel de interesse por vez e com o mesmo protocolo:

**Split dos dados.** O conjunto de dados deve ser dividido em treino e teste, com 80% das notas para treino e 20% para teste.
A divisÃ£o deve ser feita de forma estratificada por usuÃ¡rio, garantindo que cada usuÃ¡rio tenha pelo menos uma nota em cada conjunto, e todo filme deve ter pelo menos uma nota no conjunto de treino.

**MÃ©tricas.** RMSE (principal) e MAE, com previsÃµes truncadas em $[1,5]$, e NLL quando o modelo define uma verossimilhanÃ§a comparÃ¡vel.

**SeleÃ§Ã£o de hiperparÃ¢metros.** O conjunto de teste Ã© usado uma Ãºnica vez, para reportar o desempenho final.
Os hiperparÃ¢metros (dimensÃ£o latente, regularizaÃ§Ã£o, variÃ¢ncia do ruÃ­do, nÃºmero de Ã©pocas e assim por diante) sÃ£o escolhidos por *cross-validation* com $K$ folds dentro do conjunto de treino, respeitando as mesmas restriÃ§Ãµes da divisÃ£o de treino e teste.
Depois da escolha, o modelo Ã© treinado com todo o treino e avaliado no teste.

**Incerteza.** Compare resultados com **intervalos de confianÃ§a** sobre as mÃ©tricas.
VocÃªs podem usar *cross-validation*, reportando a mÃ©dia e o desvio padrÃ£o das mÃ©tricas entre os folds, que refletem a variaÃ§Ã£o causada pelos dados de treino e pela inicializaÃ§Ã£o, ou *bootstrapping*, amostrando com reposiÃ§Ã£o o conjunto de teste e reportando a mÃ©dia e o intervalo de percentis de 95%, que reflete a variaÃ§Ã£o da amostra de teste.

In [ ]:
def split_ratings(
    ratings: pd.DataFrame,
    test_frac: float = 0.2,
    seed: int = 0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = (
        ratings.sample(frac=1, random_state=seed)
               .sort_values("user_id", kind="stable")
               .reset_index(drop=True)
    )

    grouped = df.groupby("user_id")
    pos = (
        grouped.cumcount() /
        grouped["user_id"].transform("count")
    ).to_numpy()

    splits = np.zeros(len(df), dtype=np.int8)
    splits[pos >= (1 - test_frac)] = 1

    # Trocar a primeira apariÃ§Ã£o de cada filme para o conjunto de treino
    first_appearance = df.groupby("item_id").head(1).index
    splits[first_appearance] = 0

    train_df = df.iloc[splits == 0]
    test_df = df.iloc[splits == 1]

    return train_df, test_df

In [ ]:
from typing import Protocol

from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
)

# FunÃ§Ãµes auxiliares

def clip_ratings(prediction: NDArray[np.float64]) -> NDArray[np.float64]:
    return np.clip(np.asarray(prediction, dtype=np.float64), 1.0, 5.0)


class Model(Protocol):
    def fit(self, train_df: DataFrame) -> None: ...

    def predict(
        self, user_id: NDArray[np.intp], item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]: ...


class ProbabilisticModel(Model, Protocol):
    def predict_log_proba(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
        ratings: NDArray[np.float64],
    ) -> NDArray[np.float64]: ...


def evaluate_model(
    model: Model | ProbabilisticModel, **dfs: DataFrame
) -> dict[str, float]:
    metrics: dict[str, float] = {}
    
    for name, df in dfs.items():
        users = df["user_id"].to_numpy(dtype=np.intp)
        items = df["item_id"].to_numpy(dtype=np.intp)
        ratings = df["rating"].to_numpy(dtype=np.float64)

        preds = model.predict(users, items)
        preds_clipped = clip_ratings(preds)

        metrics |= {
            f"{name}_rmse": root_mean_squared_error(ratings, preds_clipped),
            f"{name}_mae": mean_absolute_error(ratings, preds_clipped),
            # VocÃªs podem adicionar mais mÃ©tricas aqui
        }

        if hasattr(model, "predict_log_proba"):
            log_proba = model.predict_log_proba(users, items, ratings)
            metrics[f"{name}_nll"] = float(-np.mean(log_proba))

    return metrics



In [ ]:
train_df, test_df = split_ratings(ratings, seed=SEED)

In [ ]:
# Explorem os dados, plotem grÃ¡ficos, calculem estatÃ­sticas, para entender melhor o problema,
# os dados disponÃ­veis, a distribuiÃ§Ã£o das notas, a distribuiÃ§Ã£o de usuÃ¡rios e itens, e assim por diante.

## Etapa 1: Baselines

O primeiro passo para entender o problema Ã© estabelecer uma baseline, um modelo simples que sirva como referÃªncia da performance que podemos esperar de um modelo mais sofisticado.
Podemos simplificar a formulaÃ§Ã£o geral do problema, considerando apenas a mÃ©dia global das notas e os *biases* do usuÃ¡rio e do item, sem considerar os vetores latentes:

$$R_{ui} = \mu + b_u + b_i + \epsilon_{ui}$$

Como o modelo remove a interaÃ§Ã£o entre usuÃ¡rios e itens, ele nÃ£o Ã© capaz de capturar a estrutura latente compartilhada entre eles, mas ainda assim pode fornecer uma boa referÃªncia para avaliar o desempenho de modelos mais complexos.
[Koren, Y. et. al.](https://ieeexplore.ieee.org/document/5197422) discute este exato modelo como uma aproximaÃ§Ã£o de primeira ordem para o problema, que pode ser melhorada com a adiÃ§Ã£o de termos de segunda ordem, como os vetores latentes introduzidos na formulaÃ§Ã£o geral.
Recomenda-se a leitura do artigo para entender melhor a motivaÃ§Ã£o e interpretaÃ§Ã£o do problema e as soluÃ§Ãµes propostas.

Devido Ã  linealidade do modelo,podemos aprender os *biases* do usuÃ¡rio e do item de forma eficiente, mesmo com a matriz de notas esparsa, atravÃ©s de regressÃ£o linear regularizada.

> **Tarefa 1.1**
> Treine o modelo de baseline e avalie seu desempenho, reportando RMSE e MAE no conjunto de teste.
> Lembre-se de usar o mesmo protocolo experimental para todos os modelos para encontrar o melhor conjunto de hiperparÃ¢metros.
> ApÃ³s treinar o modelo, analise os *biases* aprendidos e discuta o que eles podem nos dizer sobre a preferÃªncia dos usuÃ¡rios e a popularidade dos itens.

In [ ]:
class BaselineModel:
    def __init__(
        self,
        user_reg: float = 10.0,
        item_reg: float = 10.0,
        n_epochs: int = 20,
    ) -> None:
        self.user_reg = user_reg
        self.item_reg = item_reg
        self.n_epochs = n_epochs

    def fit(self, train_df: pd.DataFrame) -> None:
        ratings = train_df["rating"].to_numpy(dtype=np.float64)
        users = train_df["user_id"].to_numpy(dtype=np.intp)
        items = train_df["item_id"].to_numpy(dtype=np.intp)

        self.global_mean = float(ratings.mean())
        self.user_bias = np.zeros(int(users.max()) + 1, dtype=np.float64)
        self.item_bias = np.zeros(int(items.max()) + 1, dtype=np.float64)

        n_users = len(self.user_bias)
        n_items = len(self.item_bias)

        user_counts = np.bincount(users, minlength=n_users)
        item_counts = np.bincount(items, minlength=n_items)

        for _ in range(self.n_epochs):
            # Os vieses de usuÃ¡rio e item sÃ£o atualizados alternadamente, mantendo o outro fixo.
            # cada um resolve \min_b (sum_{(u,i) in R} (r_{ui} - mu - b_u - b_i)^2 + lambda * b^2)
            # Que Ã© equivalente Ã  agrupar por usuÃ¡rio e item, e entÃ£o resolver a equaÃ§Ã£o normal para cada um.

            user_residuals = ratings - self.global_mean - self.item_bias[items]
            self.user_bias = np.bincount(
                users, weights=user_residuals, minlength=n_users
            ) / (user_counts + self.user_reg)

            item_residuals = ratings - self.global_mean - self.user_bias[users]
            self.item_bias = np.bincount(
                items, weights=item_residuals, minlength=n_items
            ) / (item_counts + self.item_reg)

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return (
            self.global_mean
            + self.user_bias[user_id]
            + self.item_bias[item_id]
        )

In [ ]:
# FaÃ§a o treinamento e a avaliaÃ§Ã£o do modelo baseline aqui

**DiscussÃ£o:**
*Escreva suas conclusÃµes aqui*

---
### Introduzindo vetores latentes

A baseline linear captura diferenÃ§as sistemÃ¡ticas entre usuÃ¡rios e itens, mas nÃ£o explica por que determinados grupos de usuÃ¡rios atribuem notas semelhantes a determinados grupos de itens.
Para investigar essa estrutura, podemos olhar para a matriz de notas como um todo e perguntar se sua variabilidade pode ser explicada por um nÃºmero reduzido de dimensÃµes.
Essa perspectiva permite explorar interaÃ§Ãµes globais nos dados e construir uma representaÃ§Ã£o mais compacta das avaliaÃ§Ãµes, codificando usuÃ¡rios e itens em um espaÃ§o latente compartilhado.

Uma maneira de investigar essa hipÃ³tese Ã© completar a matriz de notas e aplicar AnÃ¡lise de Componentes Principais (PCA, *Principal Component Analysis*).
Para isso, substituÃ­mos as notas ausentes pela mÃ©dia do respectivo item, calculada apenas com as avaliaÃ§Ãµes observadas no conjunto de treinamento.
Em seguida, o PCA identifica as principais direÃ§Ãµes de variaÃ§Ã£o da matriz completada, permitindo representar cada usuÃ¡rio e cada item como vetores em um espaÃ§o de baixa dimensionalidade.

Seja $\tilde{R}$ a matriz de notas em que cada nota ausente foi substituÃ­da por um valor de imputaÃ§Ã£o, por exemplo, a mÃ©dia do item.
O PCA com $k$ componentes aproxima $\tilde{R}$ por uma matriz de posto $k$:

$$\hat{R}_{ui} = \bar{R}_i + \mathbf{z}_u^\top \mathbf{w}_i$$

onde $\bar{R}_i$ Ã© a mÃ©dia da coluna $i$ de $\tilde{R}$, $\mathbf{z}_u$ e $\mathbf{w}_i$ sÃ£o as coordenadas do usuÃ¡rio $u$ e do item $i$ nos $k$ primeiros componentes principais.
Sob a perspectiva do PCA, um usuÃ¡rio Ã© codificado como um vetor de pesos que indica a importÃ¢ncia de cada componente principal para aquele usuÃ¡rio, que entÃ£o Ã© decodificado em uma nota para cada item, atravÃ©s da combinaÃ§Ã£o linear com a transformaÃ§Ã£o dos itens no mesmo espaÃ§o latente.

> **Tarefa 1.2**
Treine o modelo de PCA com imputaÃ§Ã£o de notas ausentes, utilizando a mÃ©dia de cada item para preencher os valores faltantes. Utilize o protocolo experimental para selecionar o nÃºmero de componentes principais $k$ e entÃ£o reporte RMSE e MAE no conjunto de teste para o valor de $k$ selecionado.
>
> Analise e discuta tambÃ©m como o desempenho varia com o nÃºmero de componentes, discutindo se o aumento da dimensionalidade melhora as previsÃµes, produz ganhos decrescentes ou degrada o desempenho.


In [ ]:
class ImputedPCA:
    def __init__(self, n_components: int = 10) -> None:
        self.n_components = n_components
        self.user_embeddings: NDArray[np.float64]
        self.item_embeddings: NDArray[np.float64]
        self.item_means: NDArray[np.float64]
        self.pca = PCA(n_components=n_components)

    def fit(self, train_df: pd.DataFrame) -> None:
        X = ratings_to_matrix(train_df)

        # ImputaÃ§Ã£o de valores ausentes com a mÃ©dia do item
        observed = ~np.isnan(X)
        global_mean = float(X[observed].mean())
        counts = observed.sum(axis=0)
        sums = np.where(observed, X, 0.0).sum(axis=0)

        self.item_means = np.divide(
            sums,
            counts,
            out=np.full(X.shape[1], global_mean),
            where=counts > 0,
        )

        rows, cols = np.where(~observed)
        X[rows, cols] = self.item_means[cols]

        user_scores = self.pca.fit_transform(X)
        self.user_embeddings = user_scores
        self.item_embeddings = self.pca.components_.T

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return (
            self.pca.mean_[item_id]
            + np.sum(
                self.user_embeddings[user_id]
                * self.item_embeddings[item_id],
                axis=1,
            )
        )

In [ ]:
# FaÃ§a o treinamento e a avaliaÃ§Ã£o do PCA imputado aqui

**DiscussÃ£o:**
*Escreva suas conclusÃµes aqui*

---

### FatoraÃ§Ã£o de matrizes (*Matrix Factorization*)

A principal limitaÃ§Ã£o do modelo anterior Ã© que ele depende de notas imputadas, que nÃ£o correspondem a avaliaÃ§Ãµes reais e podem introduzir vieses no aprendizado dos vetores latentes.
Para superar essa limitaÃ§Ã£o, devemos aprender os vetores latentes diretamente a partir das notas observadas, sem precisar completar previamente a matriz.

A fatoraÃ§Ã£o de matrizes parte da hipÃ³tese de que a matriz de notas pode ser aproximada pelo produto de duas matrizes de menor dimensÃ£o, ou seja

$$\hat R = Z W^\top,$$
em que $Z\in\mathbb R^{|U|\times k}$ e $W\in\mathbb R^{|I|\times k}$ contÃªm os vetores latentes dos usuÃ¡rios e dos itens, respectivamente.
Dessa forma, a nota prevista para um par usuÃ¡rioâ€“item Ã© dada pelo produto interno entre suas representaÃ§Ãµes:

$$\hat R_{ui}=\mathbf z_u^\top\mathbf w_i.$$

Entretanto, como vimos na baseline, parte da variaÃ§Ã£o das notas pode ser explicada por diferenÃ§as sistemÃ¡ticas entre usuÃ¡rios e itens, independentemente de suas interaÃ§Ãµes (por exemplo, alguns filmes sÃ£o mais populares que outros).
Para considerar esses efeitos, incorporamos a mÃ©dia global e os *biases* de usuÃ¡rio e item ao modelo, obtendo a formulaÃ§Ã£o geral com $f(\mathbf{z}_u, \mathbf{w}_i) = \mathbf z_u^\top\mathbf w_i$:

$$\hat R_{ui}=\mu+b_u+b_i+\mathbf z_u^\top\mathbf w_i.$$

Para aprender os parÃ¢metros, minimizamos o erro quadrÃ¡tico entre as notas observadas e as respectivas previsÃµes, considerando apenas os pares usuÃ¡rioâ€“item avaliados no conjunto de treinamento.
Introduzimos tambÃ©m regularizaÃ§Ã£o $L_2$, que penaliza valores elevados dos *biases* e das normas dos vetores latentes:

\begin{aligned}
\min_{\mu,b,Z_U,Z_I}\quad
&\sum_{(u,i)\in\Omega}
\left(r_{ui}-\mu-b_u-b_i-\mathbf z_u^\top\mathbf w_i\right)^2\\
&+\lambda_b\left(\sum_u b_u^2+\sum_i b_i^2\right)\\
&+\lambda_z\left(\sum_u\|\mathbf z_u\|_2^2+
\sum_i\|\mathbf w_i\|_2^2\right),
\end{aligned}

em que $\Omega$ Ã© o conjunto de avaliaÃ§Ãµes observadas no treinamento, enquanto $\lambda_b$ e $\lambda_z$ controlam a intensidade da regularizaÃ§Ã£o dos *biases* e dos fatores latentes, respectivamente.
Como o problema Ã© nÃ£o-convexo, sem soluÃ§Ã£o fechada, usamos descida no gradiente para atualizar os parÃ¢metros, iterativamente com base em pequenas amostras do conjunto de treinamento.

> **Tarefa 1.3**
Treine um modelo de fatoraÃ§Ã£o de matrizes utilizando regularizaÃ§Ã£o $L_2$.
Selecione o nÃºmero de dimensÃµes latentes $k$ e os coeficientes de regularizaÃ§Ã£o com base no RMSE.
> 
> Reporte RMSE e MAE no conjunto de teste para a configuraÃ§Ã£o selecionada.
Analise como o desempenho varia com $k$ e com a regularizaÃ§Ã£o.
Por fim, compare a fatoraÃ§Ã£o com *biases* e sem *biases*, mantendo o mesmo protocolo experimental, para investigar a contribuiÃ§Ã£o dos efeitos aditivos e das interaÃ§Ãµes latentes para a qualidade das previsÃµes.
>
> Siga o mesmo protocolo experimental utilizado para os modelos anteriores.


In [ ]:
class MatrixFactorization:
    user_embeddings: NDArray[np.float64]
    item_embeddings: NDArray[np.float64]
    global_mean: float
    user_bias: NDArray[np.float64]
    item_bias: NDArray[np.float64]

    def __init__(
        self,
        n_factors: int = 10,
        n_epochs: int = 20,
        learning_rate: float = 0.01,
        reg: float = 0.1,
        bias_reg: float | None = None,
        use_biases: bool = True,
    ) -> None:
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.learning_rate = learning_rate
        self.reg = reg
        self.bias_reg = reg if bias_reg is None else bias_reg
        self.use_biases = use_biases

    def fit(self, train_df: pd.DataFrame) -> None:
        ratings = train_df["rating"].to_numpy(dtype=np.float64)
        users = train_df["user_id"].to_numpy(dtype=np.intp)
        items = train_df["item_id"].to_numpy(dtype=np.intp)

        self.global_mean = float(ratings.mean())
        n_users = int(users.max()) + 1
        n_items = int(items.max()) + 1

        self.user_embeddings = np.random.normal(scale=0.1, size=(n_users, self.n_factors))
        self.item_embeddings = np.random.normal(scale=0.1, size=(n_items, self.n_factors))
        self.user_bias = np.zeros(n_users, dtype=np.float64)
        self.item_bias = np.zeros(n_items, dtype=np.float64)

        for _ in range(self.n_epochs):
            for n in np.random.permutation(len(ratings)):
                u, i, r = users[n], items[n], ratings[n]
                pred = (
                    self.global_mean
                    + self.user_bias[u]
                    + self.item_bias[i]
                    + np.dot(self.user_embeddings[u], self.item_embeddings[i])
                )
                err = r - pred

                # AtualizaÃ§Ã£o dos vieses
                if self.use_biases:
                    self.user_bias[u] += self.learning_rate * (err - self.bias_reg * self.user_bias[u])
                    self.item_bias[i] += self.learning_rate * (err - self.bias_reg * self.item_bias[i])

                user_emb_old = self.user_embeddings[u].copy()
                item_emb_old = self.item_embeddings[i].copy()

                self.user_embeddings[u] += self.learning_rate * (err * item_emb_old - self.reg * user_emb_old)
                self.item_embeddings[i] += self.learning_rate * (err * user_emb_old - self.reg * item_emb_old)

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return (
            self.global_mean
            + self.user_bias[user_id]
            + self.item_bias[item_id]
            + np.sum(
                self.user_embeddings[user_id] * self.item_embeddings[item_id],
                axis=1,
            )
        )


In [ ]:
# FaÃ§a o treinamento e a avaliaÃ§Ã£o do modelo de fatoraÃ§Ã£o de matrizes aqui

**DiscussÃ£o:**
*Escreva suas conclusÃµes aqui*

---
## Etapa 2: SuposiÃ§Ãµes sobre a estrutura latente

Na etapa anterior, investigamos modelos que estimam notas a partir de vieses aditivos e representaÃ§Ãµes latentes de usuÃ¡rios e itens.
A fatoraÃ§Ã£o de matrizes procura representaÃ§Ãµes que aproximem as avaliaÃ§Ãµes observadas, mas nÃ£o define, por si sÃ³, uma distribuiÃ§Ã£o probabilÃ­stica completa para os dados.

Nesta etapa, exploraremos uma abordagem probabilÃ­stica baseada no PCA ProbabilÃ­stico (*Probabilistic Principal Component Analysis*, PPCA), adaptada ao problema de recomendaÃ§Ã£o.
Essa formulaÃ§Ã£o descreve as avaliaÃ§Ãµes por meio de variÃ¡veis latentes e ruÃ­do gaussiano, permitindo explicitar a variÃ¢ncia do ruÃ­do, estimar a verossimilhanÃ§a dos dados e investigar como as suposiÃ§Ãµes probabilÃ­sticas afetam as previsÃµes.

Ao contrÃ¡rio das implementaÃ§Ãµes clÃ¡ssicas de PPCA e VAE, em que o encoder Ã© treinado para mapear as observaÃ§Ãµes para a distribuiÃ§Ã£o posterior, aqui, como os usuÃ¡rios do conjunto de dados sÃ£o conhecidos durante o treinamento, podemos estimar um embedding para cada um deles e armazenar essas representaÃ§Ãµes apÃ³s o ajuste do modelo.
Isso Ã© equivalente a um *encoder* que mapeia cada usuÃ¡rio para uma distribuiÃ§Ã£o posterior com o mÃ¡ximo de liberdade, sem lidar com features de usuÃ¡rio, vieses indutivos ou generalizaÃ§Ã£o para novos usuÃ¡rios.


### PCA ProbabilÃ­stico

O PCA clÃ¡ssico identifica um subespaÃ§o linear de dimensÃ£o reduzida que representa a estrutura de variÃ¢ncia dos dados. O PPCA estende essa ideia ao assumir que as observaÃ§Ãµes sÃ£o geradas por transformaÃ§Ãµes lineares de variÃ¡veis latentes gaussianas, acrescidas de ruÃ­do gaussiano.

No problema de recomendaÃ§Ã£o, podemos representar a matriz de avaliaÃ§Ãµes por $R\in\mathbb R^{n\times m}$, em que $n$ Ã© o nÃºmero de usuÃ¡rios e $m$ Ã© o nÃºmero de filmes. Cada usuÃ¡rio corresponde a uma observaÃ§Ã£o, e cada filme corresponde a uma dimensÃ£o observada. Como os usuÃ¡rios do conjunto de dados sÃ£o conhecidos durante o treinamento, podemos estimar um embedding para cada um deles e armazenar essas representaÃ§Ãµes apÃ³s o ajuste do modelo.

Seja $\mathbf z_u\in\mathbb R^k$ a variÃ¡vel latente do usuÃ¡rio $u$, e seja $\mathbf w_i\in\mathbb R^k$ o vetor de cargas associado ao filme $i$. O modelo assume que

$$
\mathbf z_u\sim\mathcal N(\mathbf 0,I_k)
$$

e que a avaliaÃ§Ã£o do usuÃ¡rio $u$ para o filme $i$ segue

$$
R_{ui}\mid\mathbf z_u
\sim
\mathcal N\left(
\mu_i+\mathbf z_u^\top\mathbf w_i,
\sigma^2
\right),
$$

em que $\mu_i$ Ã© a mÃ©dia associada ao filme $i$, $k$ Ã© a dimensÃ£o latente e $\sigma^2$ Ã© a variÃ¢ncia do ruÃ­do, considerada fixa durante o treinamento.
A mÃ©dia condicional da avaliaÃ§Ã£o Ã©, portanto,

$$
\mathbb E[R_{ui}\mid\mathbf z_u]
=
\mu_i+\mathbf z_u^\top\mathbf w_i.
$$

Essa formulaÃ§Ã£o representa as avaliaÃ§Ãµes por meio de uma estrutura linear de baixa dimensÃ£o, sem introduzir vieses aditivos especÃ­ficos de usuÃ¡rios.
A diferenÃ§a fundamental em relaÃ§Ã£o Ã  fatoraÃ§Ã£o de matrizes determinÃ­stica estÃ¡ na distribuiÃ§Ã£o probabilÃ­stica atribuÃ­da Ã s avaliaÃ§Ãµes e Ã s variÃ¡veis latentes.

O treinamento pode ser realizado pelo algoritmo EM (*Expectation-Maximization*).
Seja $\Omega_u$ o conjunto de filmes avaliados pelo usuÃ¡rio $u$ e $\Omega_i$ o conjunto de usuÃ¡rios que avaliaram o filme $i$.
Na etapa E, como o modelo Ã© linear e gaussiano, a posterior de cada usuÃ¡rio, condicionada Ã s avaliaÃ§Ãµes observadas e aos parÃ¢metros atuais, Ã© gaussiana, $p(\mathbf z_u\mid R_u)=\mathcal N(\mathbf m_u, S_u)$, com

$$
S_u=\left( I_k + \frac{1}{\sigma^2} \sum_{i\in\Omega_u} \mathbf w_i\mathbf w_i^\top\right)^{-1},
\qquad
\mathbf m_u=\frac{1}{\sigma^2} \, S_u\sum_{i\in\Omega_u}\mathbf w_i\,(r_{ui}-\mu_i).
$$

Na etapa M, atualizamos as mÃ©dias dos filmes e os vetores de cargas para maximizar a verossimilhanÃ§a esperada.
Definindo $\boldsymbol\theta_i=(\mu_i,\mathbf w_i)$ e $\boldsymbol\phi_u=(1,\mathbf z_u)$, cada filme resolve um sistema de mÃ­nimos quadrados em que as posteriores entram pelos momentos $\mathbb E[\boldsymbol\phi_u]=(1,\mathbf m_u)$ e $\mathbb E[\mathbf z_u\mathbf z_u^\top]=S_u+\mathbf m_u\mathbf m_u^\top$:

$$
\boldsymbol\theta_i=\left(\sum_{u\in\Omega_i}\mathbb E[\boldsymbol\phi_u\boldsymbol\phi_u^\top]\right)^{-1}\sum_{u\in\Omega_i}r_{ui}\,\mathbb E[\boldsymbol\phi_u].
$$

Como a matriz de avaliaÃ§Ãµes Ã© esparsa, cada usuÃ¡rio contribui para a estimaÃ§Ã£o apenas por meio dos filmes que avaliou. As avaliaÃ§Ãµes ausentes sÃ£o tratadas como dados nÃ£o observados, e nÃ£o como notas iguais a zero. ApÃ³s o treinamento, as mÃ©dias dos embeddings posteriores podem ser armazenadas como representaÃ§Ãµes dos usuÃ¡rios, permitindo realizar previsÃµes diretamente para os pares usuÃ¡rioâ€“filme.

Para uma avaliaÃ§Ã£o observada $r_{ui}$, a densidade condicional Ã©

$$
p(r_{ui}\mid\mathbf z_u)
=
\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left(
-\frac{
(r_{ui}-\mu_i-\mathbf w_i^\top\mathbf z_u)^2
}{2\sigma^2}
\right).
$$

A log-verossimilhanÃ§a conjunta das avaliaÃ§Ãµes observadas, condicionada Ã s variÃ¡veis latentes, Ã© obtida somando as contribuiÃ§Ãµes dos pares usuÃ¡rioâ€“filme observados. O prior gaussiano das variÃ¡veis latentes tambÃ©m participa do modelo probabilÃ­stico, favorecendo representaÃ§Ãµes compatÃ­veis com a distribuiÃ§Ã£o $\mathcal N(\mathbf 0,I_k)$.

Para prever a avaliaÃ§Ã£o de um par usuÃ¡rioâ€“filme, utilizamos a mÃ©dia condicional calculada com o embedding posterior estimado:

$$
\hat R_{ui}
=
\mu_i+\mathbf w_i^\top\mathbb E[\mathbf z_u\mid R_u],
$$

em que $R_u$ representa as avaliaÃ§Ãµes observadas do usuÃ¡rio $u$. Essa previsÃ£o corresponde Ã  mÃ©dia da distribuiÃ§Ã£o preditiva quando os parÃ¢metros do modelo sÃ£o considerados fixos e a incerteza sobre a variÃ¡vel latente Ã© integrada.

A variÃ¢ncia $\sigma^2$ controla o ruÃ­do residual assumido pelo modelo. Valores menores atribuem maior concentraÃ§Ã£o Ã s avaliaÃ§Ãµes em torno da mÃ©dia condicional; valores maiores representam maior dispersÃ£o. Como a variÃ¢ncia Ã© fixada durante o treinamento nesta implementaÃ§Ã£o, ela deve ser selecionada como hiperparÃ¢metro.

> **Tarefa 2.1**
Treine o modelo PPCA para a matriz esparsa de avaliaÃ§Ãµes e selecione o nÃºmero de dimensÃµes latentes $k$ e a variÃ¢ncia do ruÃ­do $\sigma^2$ usando o protocolo experimental.
Compare o RMSE com os modelos da etapa anterior e investigue se a configuraÃ§Ã£o que minimiza o RMSE tambÃ©m produz a melhor log-verossimilhanÃ§a negativa mÃ©dia (*Negative Log-Likelihood*, NLL).
>
> Discuta como a variÃ¢ncia do ruÃ­do afeta a distribuiÃ§Ã£o das avaliaÃ§Ãµes, como o prior gaussiano influencia as representaÃ§Ãµes latentes e quais sÃ£o as implicaÃ§Ãµes de estimar embeddings especÃ­ficos para os usuÃ¡rios conhecidos durante o treinamento.

In [ ]:
class PPCA:
    user_embeddings: NDArray[np.float64]
    item_embeddings: NDArray[np.float64]
    item_means: NDArray[np.float64]
    user_covariances: NDArray[np.float64]

    def __init__(
        self, n_factors: int = 10, n_epochs: int = 100, sigma2: float = 1.0
    ) -> None:
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.sigma2 = sigma2

    def fit(self, train_df: pd.DataFrame) -> None:
        X = ratings_to_matrix(train_df)
        observed = ~np.isnan(X)
        values = np.where(observed, X, 0.0)
        weights = observed.astype(np.float64)

        counts = observed.sum(axis=0)
        global_mean = values.sum() / observed.sum()
        self.item_means = np.divide(
            values.sum(axis=0),
            counts,
            out=np.full(X.shape[1], global_mean),
            where=counts > 0,
        )

        n_users, n_items = X.shape
        k = self.n_factors
        sigma2 = self.sigma2
        eye = np.eye(k)

        self.item_embeddings = np.random.normal(
            scale=0.1, size=(n_items, k)
        )

        item_outer = (
            self.item_embeddings[:, :, None]
            * self.item_embeddings[:, None, :]
        )

        for _ in range(self.n_epochs):
            centered = np.where(observed, X - self.item_means, 0.0)

            precision = eye + np.einsum("ui,ijk->ujk", weights, item_outer) / sigma2

            self.user_covariances = np.linalg.inv(precision)

            rhs = centered @ self.item_embeddings / sigma2
            self.user_embeddings = (
                self.user_covariances @ rhs[:, :, None]
            )[:, :, 0]

            user_means = np.column_stack(
                (np.ones(n_users), self.user_embeddings)
            )

            user_second = np.zeros((n_users, k + 1, k + 1))
            user_second[:, 0, 0] = 1.0
            user_second[:, 0, 1:] = self.user_embeddings
            user_second[:, 1:, 0] = self.user_embeddings
            user_second[:, 1:, 1:] = (
                self.user_covariances
                + self.user_embeddings[:, :, None]
                * self.user_embeddings[:, None, :]
            )

            A = np.einsum("ui,uab->iab", weights, user_second)

            b = values.T @ user_means
            theta = np.linalg.solve(A, b[:, :, None])[:, :, 0]

            self.item_means = theta[:, 0]
            self.item_embeddings = theta[:, 1:]

            item_outer = (
                self.item_embeddings[:, :, None]
                * self.item_embeddings[:, None, :]
            )

        centered = np.where(observed, X - self.item_means, 0.0)

        precision = eye + np.einsum("ui,ijk->ujk", weights, item_outer) / sigma2

        self.user_covariances = np.linalg.inv(precision)

        rhs = centered @ self.item_embeddings / sigma2
        self.user_embeddings = (
            self.user_covariances @ rhs[:, :, None]
        )[:, :, 0]

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return self.item_means[item_id] + np.sum(
            self.user_embeddings[user_id] * self.item_embeddings[item_id],
            axis=1,
        )

    def predict_log_proba(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
        ratings: NDArray[np.float64],
    ) -> NDArray[np.float64]:
        user_embeddings = self.user_embeddings[user_id]
        item_embeddings = self.item_embeddings[item_id]
        covariances = self.user_covariances[user_id]

        means = self.item_means[item_id] + np.sum(
            user_embeddings * item_embeddings, axis=1
        )

        variances = self.sigma2 + np.einsum(
            "ni,nij,nj->n",
            item_embeddings,
            covariances,
            item_embeddings,
        )

        return -0.5 * (
            np.log(2 * np.pi * variances)
            + (ratings - means) ** 2 / variances
        )

In [ ]:
# FaÃ§a o treinamento e a avaliaÃ§Ã£o do PPCA aqui

**DiscussÃ£o:**
*Escreva suas conclusÃµes aqui*

### Autoencoder variacional (VAE)

O PPCA modela as avaliaÃ§Ãµes por meio de uma transformaÃ§Ã£o linear de variÃ¡veis latentes gaussianas. O VAE estende essa abordagem ao permitir um decodificador nÃ£o linear e aprender uma distribuiÃ§Ã£o latente especÃ­fica para cada usuÃ¡rio.

Seja $k$ a dimensÃ£o latente.
O codificador Ã© representado por uma matriz de embeddings de usuÃ¡rios, que fornece a mÃ©dia latente $\mathbf m_u$ e uma matriz de parÃ¢metros que fornece a variÃ¢ncia diagonal $\mathbf s_u^2$ de cada usuÃ¡rio.
Assim, a distribuiÃ§Ã£o latente aproximada Ã©

$$
\mathbf z_u
\sim
\mathcal N\left(
\mathbf m_u,
\operatorname{diag}(\mathbf s_u^2)
\right),
$$
que pode ser amostrada por meio do truque de reparametrizaÃ§Ã£o:

$$
\mathbf z_u
=
\mathbf m_u+\mathbf s_u\odot\boldsymbol\epsilon,
\qquad
\boldsymbol\epsilon\sim\mathcal N(\mathbf 0,I_k).
$$

O decodificador transforma essa amostra em previsÃµes para todos os filmes atravÃ©s de uma funÃ§Ã£o $g_\theta:\mathbb R^k\to\mathbb R^m$ parametrizada por $\theta$:

$$
\hat{\mathbf R}_u = g_\theta(\mathbf z_u).
$$

Assumimos o seguinte modelo generativo:

$$
R_{ui}\mid\mathbf z_u
\sim
\mathcal N\left(g_\theta(\mathbf z_u)_i,\sigma^2\right),
$$

em que $\sigma^2$ Ã© a variÃ¢ncia do ruÃ­do das avaliaÃ§Ãµes.
Com um decodificador linear, $g_\theta(\mathbf z)=\boldsymbol\mu+W\mathbf z$, a mÃ©dia condicional das avaliaÃ§Ãµes Ã© uma funÃ§Ã£o linear da variÃ¡vel latente, como no PPCA.
Um decodificador nÃ£o linear permite representar relaÃ§Ãµes mais complexas entre as dimensÃµes latentes e as avaliaÃ§Ãµes.

O treinamento maximiza o limite inferior da log-verossimilhanÃ§a (*Evidence Lower Bound*, ELBO), somado sobre os usuÃ¡rios:

$$
\mathcal L_u
=
\mathbb E_{q_\phi(\mathbf z_u)}
\left[
-\frac{1}{2\sigma^2}
\sum_{i\in\Omega_u}
(r_{ui}-g_\theta(\mathbf z_u)_i)^2
\right] -\beta\,\mathrm{KL}\left(
q_\phi(\mathbf z_u)
\,\|\,\mathcal N(\mathbf 0,I_k)
\right),
$$
em que $\Omega_u$ Ã© o conjunto de filmes avaliados pelo usuÃ¡rio.
Foram omitidas constantes que nÃ£o dependem dos parÃ¢metros do modelo.
Com $\beta=1$, obtemos o ELBO padrÃ£o, a menos dessas constantes, variar $\beta$ altera o peso relativo da regularizaÃ§Ã£o latente.

O primeiro termo incentiva o modelo a atribuir alta probabilidade Ã s notas observadas, o segundo aproxima a distribuiÃ§Ã£o latente de cada usuÃ¡rio do prior gaussiano padrÃ£o, regularizando as representaÃ§Ãµes aprendidas.
A esperanÃ§a Ã© estimada por amostragem da distribuiÃ§Ã£o latente, permitindo propagar gradientes pelos parÃ¢metros do codificador e do decodificador.

Na prediÃ§Ã£o, podemos usar a mÃ©dia latente $\mathbf m_u$ como embedding do usuÃ¡rio e calcular

$$
\hat R_{ui} = g_\theta(\mathbf m_u)_i.
$$

Essa escolha fornece uma previsÃ£o pontual determinÃ­stica. A distribuiÃ§Ã£o preditiva completa, por sua vez, tambÃ©m considera a variabilidade latente e o ruÃ­do das avaliaÃ§Ãµes.

Diferentemente do PPCA, que calcula a posterior de cada usuÃ¡rio analiticamente a partir das avaliaÃ§Ãµes observadas, o VAE aprende por otimizaÃ§Ã£o uma distribuiÃ§Ã£o latente especÃ­fica para cada usuÃ¡rio, representada por uma mÃ©dia e uma variÃ¢ncia. Essa formulaÃ§Ã£o Ã© transdutiva: as representaÃ§Ãµes sÃ£o aprendidas para os usuÃ¡rios presentes no treinamento e nÃ£o se estendem automaticamente a usuÃ¡rios novos.

> **Tarefa 2.2**
Treine um VAE com decodificador linear e compare seu desempenho com o PPCA usando o mesmo nÃºmero de dimensÃµes latentes $k$ e a mesma variÃ¢ncia do ruÃ­do $\sigma^2$. Compare RMSE, MAE e NLL no conjunto de teste.
>
> Em seguida, substitua o decodificador linear por um nÃ£o linear e investigue se a flexibilidade adicional melhora as previsÃµes. Escolha um hiperparÃ¢metros para variar isoladamente, $\beta$, $\sigma^2$ ou $k$, e aplique o protocolo experimental.
Verifique a convergÃªncia pela curva de treinamento armazenada em `history` antes de comparar os modelos.

AtenÃ§Ã£o, com um decoder nÃ£o-linear, os itens 

In [ ]:
import torch
from torch import nn


class VAE:
    user_embeddings: NDArray[np.float64]
    user_variances: NDArray[np.float64]
    item_embeddings: NDArray[np.float64]
    item_means: NDArray[np.float64]

    def __init__(
        self,
        n_factors: int = 10,
        n_epochs: int = 300,
        sigma2: float = 1.0,
        beta: float = 1.0,
        hidden_dim: int | None = None,
        learning_rate: float = 1e-2,
    ) -> None:
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.sigma2 = sigma2
        self.beta = beta
        self.hidden_dim = hidden_dim
        self.learning_rate = learning_rate
        self.history: list[float] = []

    def fit(self, train_df: pd.DataFrame) -> None:
        X = ratings_to_matrix(train_df)
        observed = ~np.isnan(X)
        values = np.where(observed, X, 0.0)

        X = torch.tensor(values, dtype=torch.float32)
        mask = torch.tensor(observed, dtype=torch.float32)

        n_users, n_items = X.shape
        k = self.n_factors

        self.user_means = nn.Parameter(torch.randn(n_users, k) * 0.01)
        self.user_logvars = nn.Parameter(torch.zeros(n_users, k))

        if self.hidden_dim is None:
            self.decoder = nn.Linear(k, n_items)

        else:
            self.decoder = nn.Sequential(
                nn.Linear(k, self.hidden_dim),
                nn.Tanh(),
                nn.Linear(self.hidden_dim, n_items),
            )

        # O viÃ©s do decodificador comeÃ§a nas mÃ©dias dos filmes, e o treino modela o desvio em relaÃ§Ã£o a elas
        counts = np.maximum(observed.sum(axis=0), 1)
        item_means = np.where(observed.any(axis=0), values.sum(axis=0) / counts, values.sum() / observed.sum())
        output_layer = self.decoder if self.hidden_dim is None else self.decoder[-1]
        with torch.no_grad():
            output_layer.bias.copy_(torch.tensor(item_means, dtype=torch.float32))

        parameters = [
            self.user_means,
            self.user_logvars,
            *self.decoder.parameters(),
        ]
        optimizer = torch.optim.Adam(parameters, lr=self.learning_rate)

        self.history = []

        for _ in range(self.n_epochs):
            optimizer.zero_grad()

            std = torch.exp(0.5 * self.user_logvars)
            epsilon = torch.randn_like(std)
            z = self.user_means + std * epsilon

            predictions = self.decoder(z)

            log_likelihood = -0.5 * (
                mask
                * (
                    np.log(2 * np.pi * self.sigma2)
                    + (X - predictions) ** 2 / self.sigma2
                )
            ).sum()

            kl = -0.5 * (
                1
                + self.user_logvars
                - self.user_means**2
                - self.user_logvars.exp()
            ).sum()

            elbo = log_likelihood - self.beta * kl
            loss = -elbo / n_users

            loss.backward()
            optimizer.step()

            self.history.append((-loss.detach()).item())

        with torch.no_grad():
            self.user_embeddings = (
                self.user_means.detach().cpu().numpy().astype(np.float64)
            )
            self.user_variances = (
                self.user_logvars.exp()
                .detach()
                .cpu()
                .numpy()
                .astype(np.float64)
            )

            if self.hidden_dim is None:
                self.item_embeddings = (
                    self.decoder.weight.detach()
                    .cpu().numpy().astype(np.float64)
                )
                self.item_means = (
                    self.decoder.bias.detach()
                    .cpu().numpy().astype(np.float64)
                )

    def _decode_at(self, z: torch.Tensor, items: torch.Tensor) -> torch.Tensor:
        """Decode only the entries (z[..., n, :], items[n]), without computing every movie."""
        if self.hidden_dim is None:
            return (z * self.decoder.weight[items]).sum(-1) + self.decoder.bias[items]
        first, _, last = self.decoder
        return (torch.tanh(first(z)) * last.weight[items]).sum(-1) + last.bias[items]

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        users = torch.as_tensor(user_id, dtype=torch.long)
        items = torch.as_tensor(item_id, dtype=torch.long)

        with torch.no_grad():
            z = torch.as_tensor(self.user_embeddings, dtype=torch.float32)[users]
            result = self._decode_at(z, items)

        return result.cpu().numpy().astype(np.float64)

    def predict_log_proba(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
        ratings: NDArray[np.float64],
        n_samples: int = 100,
        chunk: int = 2048,
    ) -> NDArray[np.float64]:
        """Log of the Monte Carlo predictive density, computed in chunks of rating pairs."""
        users = torch.as_tensor(user_id, dtype=torch.long)
        items = torch.as_tensor(item_id, dtype=torch.long)
        ratings_t = torch.as_tensor(ratings, dtype=torch.float32)
        user_means = torch.as_tensor(self.user_embeddings, dtype=torch.float32)
        user_stds = torch.as_tensor(self.user_variances, dtype=torch.float32).sqrt()

        result = []
        with torch.no_grad():
            for start in range(0, len(users), chunk):
                u = users[start : start + chunk]
                means, stds = user_means[u], user_stds[u]
                z = means + stds * torch.randn(n_samples, *means.shape)
                predictions = self._decode_at(z, items[start : start + chunk])

                log_probs = -0.5 * (
                    np.log(2 * np.pi * self.sigma2)
                    + (ratings_t[start : start + chunk] - predictions) ** 2 / self.sigma2
                )
                result.append(torch.logsumexp(log_probs, dim=0) - np.log(n_samples))

        return torch.cat(result).cpu().numpy().astype(np.float64)

In [ ]:
# FaÃ§a o treinamento e a avaliaÃ§Ã£o do VAE aqui

**DiscussÃ£o:**
*Escreva suas conclusÃµes aqui*

---

### VisualizaÃ§Ã£o dos espaÃ§os latentes

Dos modelos treinados, quatro deles aprendem vetores latentes por usuÃ¡rio e por item: PCA com imputaÃ§Ã£o, fatoraÃ§Ã£o de matrizes, PPCA e VAE.
Esses vetores podem ser analisados com as mesmas ferramentas, permitindo que possamos interpretar e comparar os espaÃ§os latentes aprendidos por cada modelo.
Garante-se que pelo menos um modelo de cada tipo tenha sido treinado, e entÃ£o podemos comparar os espaÃ§os latentes aprendidos.

> **Tarefa 2.3**
Visualize os espaÃ§os latentes de usuÃ¡rios e de itens dos quatro modelos, usando um $k$ de sua escolha, e relacione o que aparece nas figuras com informaÃ§Ãµes que nÃ£o foram usadas no treino: por exemplo, gÃªnero, idade e ocupaÃ§Ã£o dos usuÃ¡rios, e gÃªneros, ano e popularidade dos filmes.
>
> O espaÃ§o de cada modelo organiza essas informaÃ§Ãµes, e isso pode ser medido alÃ©m da inspeÃ§Ã£o visual?
Os quatro espaÃ§os compartilham a mesma estrutura, ou cada suposiÃ§Ã£o produz uma geometria diferente?


**Ideias para comeÃ§ar.**

Para visualizar, projete os vetores em duas dimensÃµes com PCA, que Ã© determinÃ­stico e preserva a geometria global, e compare com o t-SNE.
Colorir os pontos por um metadado mostra se ele aparece, e um modelo supervisionado linear (`Ridge` para variÃ¡veis contÃ­nuas, `LogisticRegression` com `roc_auc_score` para categorias) pode medir quanto de informaÃ§Ã£o sobre o metadado estÃ¡ presente no espaÃ§o latente original, sem precisar de uma projeÃ§Ã£o.

Isso permite verificar se, a partir dos vetores latentes, Ã© possÃ­vel prever a idade do usuÃ¡rio, o gÃªnero do filme ou a popularidade do filme, quantificando o quanto de informaÃ§Ã£o sobre essas variÃ¡veis estÃ¡ presente no espaÃ§o latente.

Ao comparar os espaÃ§os latentes, Ã© importante lembrar que cada modelo aprende uma representaÃ§Ã£o diferente, e que a correspondÃªncia entre os vetores latentes de um modelo e de outro nÃ£o Ã© trivial, pois os espaÃ§os latentes podem ser invariantes Ã  rotaÃ§Ã£o, reflexÃ£o e escala.
Para comparar dois espaÃ§os Ã© necessÃ¡rio alinhar os vetores latentes de um modelo com os do outro, ou atravÃ©s de anÃ¡lises de correspondÃªncia de vizinhos, como a correlaÃ§Ã£o de distÃ¢ncias entre pontos, ou atravÃ©s da sobreposiÃ§Ã£o dos vizinhos mais prÃ³ximos de cada filme em dois modelos.

In [ ]:
# FaÃ§a as visualizaÃ§Ãµes e anÃ¡lises dos espaÃ§os latentes aqui

**DiscussÃ£o:**
*Escreva suas conclusÃµes aqui*

---

## Etapa 3: Projeto do grupo

Nesta Ãºltima etapa, cada um de vocÃªs deve propor um projeto de investigaÃ§Ã£o, que pode ser uma melhoria do modelo ou uma anÃ¡lise mais profunda de algum aspecto do problema.
ApÃ³s finalizar as duas primeiras etapas, verifiquem se hÃ¡ alguma pergunta que surgiu durante as anÃ¡lises e ainda nÃ£o foi explorado, ou alguma hipÃ³tese que vocÃªs gostariam de testar, por exemplo, sobre a geometria do espaÃ§o latente, uma formulaÃ§Ã£o alternativa, ou um hiperparÃ¢metro que nÃ£o foi isolado.

O importante Ã© que o projeto esteja bem definido, com uma motivaÃ§Ã£o clara, uma hipÃ³tese testÃ¡vel e um protocolo experimental que permita avaliar o impacto da mudanÃ§a proposta.

> **Tarefa 3.1**
Escolha um projeto e escreva a pergunta, a hipÃ³tese, a previsÃ£o e o experimento antes de rodar.
Defina o modelo referÃªncia, qual Ã© a mudanÃ§a proposta e qual resultado refutaria a hipÃ³tese de vocÃªs.
Ao final, compare o que foi previsto com o que foi observado e o que o resultado diz sobre a previsÃ£o sobre o espaÃ§o latente.

**Projetos sugeridos.**
Aqui estÃ¡ uma lista de ideias que podem ser exploradas, mas vocÃªs podem propor outras, desde que estejam bem justificadas.

*1. AusÃªncia nÃ£o aleatÃ³ria.*
UsuÃ¡rios ativos e filmes populares aparecem mais nos dados, o que cria um viÃ©s de observaÃ§Ã£o.
Pergunta: corrigir esse viÃ©s com pesos pelo inverso da probabilidade de observaÃ§Ã£o [Schnabel et al.](https://arxiv.org/abs/1602.05352) melhora as previsÃµes para filmes pouco avaliados, e o que acontece com os populares?

*2. Metadados como features.*
Ao invÃ©s de tratar cada usuÃ¡rio e item como um identificador Ãºnico, podemos usar suas caracterÃ­sticas como features para o "codificador".
Pergunta: usar os metadados para definir a mÃ©dia do prior de $\mathbf{z}$ melhora as previsÃµes de usuÃ¡rios com pouco histÃ³rico?

*3. Geometria da compatibilidade.*
Todos os modelos utilizam o produto interno para medir a compatibilidade entre usuÃ¡rios e itens.
Pergunta: trocar o produto interno por uma distÃ¢ncia, ou por uma pequena rede que recebe $\mathbf{z}_u$ e $\mathbf{w}_i$, muda o que o espaÃ§o codifica e o erro de previsÃ£o? Ver [Rendle et al.](https://arxiv.org/abs/2005.09683) para um resultado que motiva a pergunta.

*4. Estrutura do espaÃ§o latente.*
A selecÃ£o de hiperparÃ¢metros nÃ£o observou a geometria do espaÃ§o latente, mas apenas o erro de previsÃ£o.
Pergunta: o que muda no espaÃ§o latente aprendido do VAE ao variar $\beta$, o tamanho do espaÃ§o ou a regularizaÃ§Ã£o, em termos de estabilidade de clusters, correlaÃ§Ã£o com metadados e erro de previsÃ£o?

In [ ]:
# Desenvolva o experimento do seu grupo aqui

**DiscussÃ£o:**
*Escreva suas conclusÃµes aqui*